# SWaT Teacher Representation Benchmark: T4 vs T5
### Comparing Process-Stage Coupled VLM (T4) vs. Vision Foundation Model (T5)

Evaluates:
1. **T4: Process-Stage Coupled VLM Teacher** (`Qwen2.5-VL-3B` with Stage-Coupled channels and single-pass logit-difference scoring).
2. **T5: Vision Foundation Model** (`ViT-B/16` deep patch embeddings + cosine anomaly score).


In [ ]:
# --- Step 1: Workspace Setup & Codebase Initialization ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = "/kaggle/working/VisTAD"
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print("Unpacking self-contained VisTAD codebase (src, vlm4ts, configs, runners)...")
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip), "r") as z:
    z.extractall(repo_dir)
print("VisTAD codebase unpacked successfully.")
print("Extracted files:", os.listdir(repo_dir))


In [ ]:
# --- Step 2: GPU Accelerator Diagnostics ---
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device:     {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No CUDA GPU detected! Ensure GPU is selected in Kaggle settings.")


In [ ]:
# --- Step 3: Dataset Auto-Discovery ---
import os, glob, shutil
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Scanning /kaggle/input for dataset files...")
for root, dirs, files in os.walk("/kaggle/input"):
    for ds in ["swat", "smd", "msl", "gecco"]:
        if ds in dirs and not os.path.exists(f"data/{ds}"):
            src = os.path.join(root, ds)
            try:
                os.symlink(src, f"data/{ds}")
            except Exception:
                shutil.copytree(src, f"data/{ds}")

print("Active datasets in data/:", [d for d in ["swat", "smd", "msl", "gecco"] if os.path.exists(f"data/{d}")])


In [ ]:
# --- Step 4: Execute Teacher Benchmark (T4 vs T5) on SWaT ---
import os
cmd = "python vlm4ts/teacher_evaluator.py --n_eval 100 --output_md results/teacher_t4_t5_comparison.md --output_csv results/teacher_t4_t5_comparison.csv"
ret = os.system(cmd)
assert ret == 0, f"Teacher evaluation failed with return code {ret}"


In [ ]:
# --- Step 5: Display Comparison Table & Results ---
import pandas as pd
df = pd.read_csv("results/teacher_t4_t5_comparison.csv")
print("\n" + "="*85)
print("                  SWAT TEACHER REPRESENTATION BENCHMARK RESULTS")
print("="*85)
print(df.to_string(index=False))
print("="*85 + "\n")


In [ ]:
# --- Step 6: Package Results Bundle ---
import os, zipfile, glob
bundle_path = "/kaggle/working/teacher_t4_t5_bundle.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["results/teacher_t4_t5_comparison.md", "results/teacher_t4_t5_comparison.csv"]:
        if os.path.exists(f):
            zf.write(f)
print(f"Bundle created: {bundle_path} ({os.path.getsize(bundle_path)} bytes)")
